In [ ]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# Run once per Colab session
# ============================================================
!pip install -q snowflake-connector-python transformers peft
!pip install -q accelerate bitsandbytes datasets trl
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# ============================================================
# CELL 2 — CONFIGURATION
# Set all parameters here before running any other cells
# ============================================================

# Snowflake connection
SF_ACCOUNT   = "your_account"       # e.g. abc12345.us-east-1
SF_USER      = "your_user"
SF_PASSWORD  = "your_password"
SF_WAREHOUSE = "COMPUTE_WH"
SF_DATABASE  = "AIRFLOW_ML"

# Snowflake table references
ISSUES_TABLE = "AIRFLOW_ML.CLEANED.GITHUB_ISSUES"
PRS_TABLE    = "AIRFLOW_ML.RAW.GITHUB_PRS"

# Training set targets
YES_TARGET   = 2700   # issues WITH merged PR → REQUIRES_CODE_CHANGE: YES
NO_TARGET    = 300    # issues WITHOUT any PR → REQUIRES_CODE_CHANGE: NO

# Patcher filter thresholds (Planner uses any merged PR, not filtered)
# For Planner we only need the file list, not the diff — so no patch size filter
# We DO prefer PRs with fewer files for cleaner plans — soft preference only
PREFER_MAX_FILES = 10   # prefer PRs with <= this many files (soft filter)

# Model config
BASE_MODEL       = "Qwen/Qwen2.5-Coder-7B-Instruct"
OUTPUT_DIR       = "/content/planner_output"
JSONL_PATH       = "/content/planner_training.jsonl"
CSV_BACKUP_PATH  = "/content/planner_pairs.csv"

# Tree-sitter index — upload treesitter_index.json to Colab then set path
TREESITTER_INDEX_PATH = "/content/treesitter_index.json"

# Training hyperparameters
LORA_R           = 32
LORA_ALPHA       = 64
LORA_DROPOUT     = 0.05
LORA_TARGET_MODS = ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj"]
MAX_SEQ_LEN      = 2048
EPOCHS           = 3
BATCH_SIZE       = 8
GRAD_ACCUM       = 2       # effective batch = 8 * 2 = 16
LR               = 1e-4
WARMUP_RATIO     = 0.05

# Loss weight for NO examples (90-10 split — upweight minority class)
NO_LOSS_WEIGHT   = 2.5

# Random seed for reproducibility
SEED = 42

print("Config loaded.")

In [ ]:
# ============================================================
# CELL 3 — IMPORTS
# ============================================================
import json
import re
import csv
import random
import logging
import warnings
from pathlib import Path
from collections import defaultdict

import snowflake.connector
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
from datasets import Dataset

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO)
random.seed(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# CELL 4 — SNOWFLAKE CONNECTION
# ============================================================
def get_sf_conn():
    return snowflake.connector.connect(
        account=SF_ACCOUNT,
        user=SF_USER,
        password=SF_PASSWORD,
        warehouse=SF_WAREHOUSE,
        database=SF_DATABASE,
    )

# Test connection
conn = get_sf_conn()
cur = conn.cursor()
cur.execute("SELECT CURRENT_TIMESTAMP()")
print("Snowflake connected:", cur.fetchone()[0])
conn.close()

In [ ]:
# ============================================================
# CELL 5 — LOAD TREE-SITTER INDEX
# Upload treesitter_index.json to Colab before running this cell
# Built by running build_treesitter_index.py locally on your Airflow clone
# ============================================================

if not Path(TREESITTER_INDEX_PATH).exists():
    # Fallback: empty index — Planner will train without code symbol context
    # Replace with real index for best model quality
    print("WARNING: treesitter_index.json not found. Training without symbol context.")
    print("Upload the file from build_treesitter_index.py output and rerun.")
    TREESITTER_INDEX = {}
else:
    with open(TREESITTER_INDEX_PATH) as f:
        TREESITTER_INDEX = json.load(f)
    print(f"Tree-sitter index loaded: {len(TREESITTER_INDEX)} files")
    total_syms = sum(len(v['classes']) + len(v['functions']) for v in TREESITTER_INDEX.values())
    print(f"Total symbols: {total_syms}")

In [ ]:
# ============================================================
# CELL 6 — TREE-SITTER HELPER FUNCTIONS
# These replicate the logic from build_treesitter_index.py
# so the same subset extraction works at training AND production inference
# ============================================================

def extract_keywords(text: str) -> set:
    """Extract keywords from issue text to match against the symbol index."""
    keywords = set()
    # CamelCase identifiers
    camel = re.findall(r'\b[A-Z][a-zA-Z0-9]+\b', text)
    keywords.update(w.lower() for w in camel)
    # snake_case identifiers
    snake = re.findall(r'\b[a-z][a-z0-9]*(?:_[a-z0-9]+){1,}\b', text.lower())
    keywords.update(snake)
    # airflow/ file path mentions
    paths = re.findall(r'airflow/[a-zA-Z0-9_/]+\.py', text)
    keywords.update(p.lower() for p in paths)
    # Component keywords
    for kw in ["scheduler", "executor", "dag", "task", "operator", "sensor",
               "hook", "provider", "trigger", "xcom", "variable", "connection",
               "webserver", "api", "model", "database"]:
        if kw in text.lower():
            keywords.add(kw)
    return keywords


def get_relevant_index_subset(issue_text: str, max_files: int = 40) -> dict:
    """Get the relevant slice of the Tree-sitter index for a given issue."""
    if not TREESITTER_INDEX:
        return {}
    keywords = extract_keywords(issue_text)
    if not keywords:
        return {}
    scored = []
    for file_path, symbols in TREESITTER_INDEX.items():
        score = 0
        file_lower = file_path.lower()
        for kw in keywords:
            if kw in file_lower:
                score += 2
        for sym in symbols.get("classes", []) + symbols.get("functions", []):
            for kw in keywords:
                if kw in sym.lower():
                    score += 1
        if score > 0:
            scored.append((score, file_path, symbols))
    scored.sort(reverse=True, key=lambda x: x[0])
    return {fp: syms for _, fp, syms in scored[:max_files]}


def format_index_for_prompt(subset: dict) -> str:
    """Format a subset of the index as a compact string for the model prompt."""
    lines = []
    for file_path, symbols in subset.items():
        parts = []
        if symbols.get("classes"):
            parts.append(f"classes: {', '.join(symbols['classes'][:5])}")
        if symbols.get("functions"):
            parts.append(f"functions: {', '.join(symbols['functions'][:8])}")
        if parts:
            lines.append(f"{file_path} | {' | '.join(parts)}")
    return "\n".join(lines)


print("Tree-sitter helpers loaded.")

In [ ]:
# ============================================================
# CELL 7 — PULL YES EXAMPLES FROM SNOWFLAKE
# Issues with at least one linked merged PR
# For Planner: we use ALL merged PRs (no file count filter here)
# We prefer PRs with fewer files (soft sort) for cleaner plan supervision
# Ground truth: pr.body + pr_files[].filename + first commit message
# ============================================================

conn = get_sf_conn()
cur = conn.cursor()

# Pull issues with linked merged PRs
# Join on LINKED_ISSUE_NUMBER to get the best PR per issue
# Best = merged PR with fewest files (cleanest plan signal)
# If multiple merged PRs exist per issue, prefer the one with fewer files
YES_QUERY = f"""
    WITH ranked_prs AS (
        SELECT
            P.LINKED_ISSUE_NUMBER,
            P.PR_NUMBER,
            P.PR_TITLE,
            P.CHANGED_FILES_COUNT,
            P.RAW_JSON AS PR_RAW,
            ROW_NUMBER() OVER (
                PARTITION BY P.LINKED_ISSUE_NUMBER
                ORDER BY P.CHANGED_FILES_COUNT ASC, P.PR_NUMBER DESC
            ) AS rn
        FROM {PRS_TABLE} P
        WHERE P.IS_MERGED = TRUE
          AND P.LINKED_ISSUE_NUMBER IS NOT NULL
    )
    SELECT
        I.ISSUE_NUMBER,
        I.TITLE,
        I.LABELS,
        I.ASSIGNEE_COUNT,
        I.RAW_JSON AS ISSUE_RAW,
        R.PR_NUMBER,
        R.PR_TITLE,
        R.CHANGED_FILES_COUNT,
        R.PR_RAW
    FROM {ISSUES_TABLE} I
    JOIN ranked_prs R
        ON R.LINKED_ISSUE_NUMBER = I.ISSUE_NUMBER
        AND R.rn = 1
    WHERE I.STATE = 'closed'
      AND I.CLOSED_AT IS NOT NULL
    ORDER BY RANDOM()
    LIMIT {YES_TARGET * 2}  -- pull 2x, we'll filter below
"""

cur.execute(YES_QUERY)
yes_rows = cur.fetchall()
yes_cols = [d[0] for d in cur.description]
yes_df = pd.DataFrame(yes_rows, columns=yes_cols)
print(f"YES candidates pulled: {len(yes_df)}")
print(f"Avg files changed: {yes_df['CHANGED_FILES_COUNT'].mean():.1f}")
print(f"PRs with <= 6 files: {(yes_df['CHANGED_FILES_COUNT'] <= 6).sum()}")
print(f"PRs with 7-10 files: {((yes_df['CHANGED_FILES_COUNT'] > 6) & (yes_df['CHANGED_FILES_COUNT'] <= 10)).sum()}")
print(f"PRs with > 10 files: {(yes_df['CHANGED_FILES_COUNT'] > 10).sum()}")

conn.close()

In [ ]:
# ============================================================
# CELL 8 — PULL NO EXAMPLES FROM SNOWFLAKE
# Issues without any linked PR — REQUIRES_CODE_CHANGE: NO
# Random sample from the 4348 no-PR issues
# Spot-check 10 before proceeding
# ============================================================

conn = get_sf_conn()
cur = conn.cursor()

NO_QUERY = f"""
    SELECT
        I.ISSUE_NUMBER,
        I.TITLE,
        I.LABELS,
        I.ASSIGNEE_COUNT,
        I.RAW_JSON AS ISSUE_RAW
    FROM {ISSUES_TABLE} I
    WHERE I.LINKED_PR_COUNT = 0
      AND I.STATE = 'closed'
      AND I.CLOSED_AT IS NOT NULL
    ORDER BY RANDOM()
    LIMIT {NO_TARGET * 2}  -- pull 2x for manual spot-check filtering
"""

cur.execute(NO_QUERY)
no_rows = cur.fetchall()
no_cols = [d[0] for d in cur.description]
no_df = pd.DataFrame(no_rows, columns=no_cols)
print(f"NO candidates pulled: {len(no_df)}")
conn.close()

# ---- SPOT CHECK: print 10 NO examples for manual review ----
# Review these titles before proceeding
# If any obviously describe a code bug (not a doc/question/duplicate), note the index
print("\n=== SPOT CHECK: 10 NO examples — review titles before proceeding ===")
sample_no = no_df.sample(10, random_state=SEED)
for i, row in enumerate(sample_no.itertuples()):
    raw = json.loads(row.ISSUE_RAW)
    issue = raw.get('issue', {})
    labels = [l.get('name','') for l in issue.get('labels', [])]
    state_reason = issue.get('state_reason', '')
    print(f"  [{i+1}] #{row.ISSUE_NUMBER}: {row.TITLE[:80]}")
    print(f"       labels={labels} | state_reason={state_reason}")

In [ ]:
# ============================================================
# CELL 9 — BUILD TRAINING PAIRS
# Construct (input_prompt, target_output) pairs for each example
# YES: structured plan derived from PR body + file list + commit message
# NO: short output ending at REQUIRES_CODE_CHANGE: NO + REASON
# Closure fields are NEVER included in the input prompt
# ============================================================

TOKENIZER = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)


def truncate_text(text: str, max_tokens: int) -> str:
    """Truncate text to approximately max_tokens using char estimate (4 chars/token)."""
    max_chars = max_tokens * 4
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "...[truncated]"


def count_tokens(text: str) -> int:
    """Count tokens using the actual tokenizer."""
    return len(TOKENIZER.encode(text, add_special_tokens=False))


def build_yes_pair(row) -> dict | None:
    """
    Build a YES training pair from an issue + its best merged PR.

    Input  = issue context (no closure fields) + tree-sitter index
    Output = structured plan with REQUIRES_CODE_CHANGE: YES header

    Ground truth for the plan comes from:
      - pr.body (explains what was done and why)
      - pr_files[].filename (which files were actually touched)
      - commits[0].commit.message (root cause in developer's own words)
    """
    try:
        issue_raw = json.loads(row.ISSUE_RAW)
        pr_raw    = json.loads(row.PR_RAW)
    except Exception:
        return None

    issue    = issue_raw.get('issue', {})
    comments = issue_raw.get('comments', [])
    pr       = pr_raw.get('pr', {})
    files    = pr_raw.get('files', [])
    commits  = pr_raw.get('commits', [])

    # ---- Build INPUT ----
    title        = (issue.get('title') or '').strip()
    body         = truncate_text((issue.get('body') or '').strip(), 800)
    labels       = ', '.join(l.get('name','') for l in issue.get('labels', []) if l.get('name'))
    assignee_cnt = len(issue.get('assignees') or [])

    # Full comment thread (usually short, don't truncate)
    comment_text = ' | '.join(
        (c.get('body') or '').strip()[:300]
        for c in comments
        if c.get('body','').strip()
    )

    # Tree-sitter index subset for this issue
    search_text  = f"{title} {body} {comment_text}"
    ts_subset    = get_relevant_index_subset(search_text, max_files=35)
    ts_text      = format_index_for_prompt(ts_subset)
    ts_text      = truncate_text(ts_text, 400)  # cap index at 400 tokens

    # NOTE: No closure fields — no closed_at, state_reason, closed_by, updated_at
    input_prompt = (
        "You are a senior software engineer planning a code patch for an open GitHub issue.\n\n"
        f"ISSUE: {title}\n"
        f"LABELS: {labels}\n"
        f"ASSIGNEES: {assignee_cnt}\n"
        f"BODY: {body}\n"
        f"COMMENTS: {comment_text[:600] if comment_text else 'none'}\n"
        f"REPO_SYMBOLS:\n{ts_text}\n\n"
        "First decide if a code change is needed. If yes, produce a structured patch plan."
    )

    # ---- Build OUTPUT (ground truth from merged PR) ----
    pr_body      = (pr.get('body') or '').strip()
    file_list    = [f.get('filename','') for f in files if f.get('filename')]
    first_commit = ''
    if commits:
        first_commit = (commits[0].get('commit',{}).get('message') or '').strip().split('\n')[0]

    # Derive root cause from first commit message (developer's own words)
    root_cause = first_commit or truncate_text(pr_body, 80)

    # Derive target files — all files in the PR (this IS the ground truth)
    target_files = '\n  - '.join(file_list[:10])  # cap display at 10
    if len(file_list) > 10:
        target_files += f"\n  - ... and {len(file_list) - 10} more"

    # Patch strategy from PR body (first 200 chars is usually the summary)
    patch_strategy = truncate_text(pr_body, 200).replace('\n', ' ').strip()

    target_output = (
        "REQUIRES_CODE_CHANGE: YES\n"
        f"CONFIDENCE: HIGH\n"
        f"REASON: {root_cause}\n\n"
        "PLAN:\n"
        f"- Root cause: {root_cause}\n"
        f"- Target files:\n  - {target_files}\n"
        f"- Patch strategy: {patch_strategy}\n"
        "- Test strategy: Update or add unit tests for the modified functions."
    )

    # ---- Token budget check ----
    full_text   = f"<|im_start|>user\n{input_prompt}<|im_end|>\n<|im_start|>assistant\n{target_output}<|im_end|>"
    token_count = count_tokens(full_text)

    if token_count > MAX_SEQ_LEN:
        # Retry with harder truncation on body and comment
        body         = truncate_text(body, 400)
        comment_text = comment_text[:400]
        ts_text      = truncate_text(ts_text, 200)
        input_prompt = (
            "You are a senior software engineer planning a code patch for an open GitHub issue.\n\n"
            f"ISSUE: {title}\n"
            f"LABELS: {labels}\n"
            f"ASSIGNEES: {assignee_cnt}\n"
            f"BODY: {body}\n"
            f"COMMENTS: {comment_text if comment_text else 'none'}\n"
            f"REPO_SYMBOLS:\n{ts_text}\n\n"
            "First decide if a code change is needed. If yes, produce a structured patch plan."
        )
        full_text   = f"<|im_start|>user\n{input_prompt}<|im_end|>\n<|im_start|>assistant\n{target_output}<|im_end|>"
        token_count = count_tokens(full_text)

        # Hard discard if still over limit
        if token_count > MAX_SEQ_LEN:
            return None

    return {
        'issue_number': row.ISSUE_NUMBER,
        'pr_number':    row.PR_NUMBER,
        'label':        'YES',
        'input':        input_prompt,
        'output':       target_output,
        'full_text':    full_text,
        'token_count':  token_count,
        'changed_files': row.CHANGED_FILES_COUNT,
    }


def build_no_pair(row) -> dict | None:
    """
    Build a NO training pair from an issue without a linked PR.

    Input  = issue context (same format as YES)
    Output = REQUIRES_CODE_CHANGE: NO + REASON (output stops here, no PLAN)

    Loss weight for NO examples is 2.5x (configured in training loop)
    to compensate for the 90-10 YES/NO imbalance.
    """
    try:
        issue_raw = json.loads(row.ISSUE_RAW)
    except Exception:
        return None

    issue    = issue_raw.get('issue', {})
    comments = issue_raw.get('comments', [])

    title        = (issue.get('title') or '').strip()
    body         = truncate_text((issue.get('body') or '').strip(), 800)
    labels       = ', '.join(l.get('name','') for l in issue.get('labels', []) if l.get('name'))
    assignee_cnt = len(issue.get('assignees') or [])
    comment_text = ' | '.join(
        (c.get('body') or '').strip()[:300]
        for c in comments
        if c.get('body','').strip()
    )

    # Tree-sitter index (same as YES — model sees same input format)
    search_text  = f"{title} {body}"
    ts_subset    = get_relevant_index_subset(search_text, max_files=35)
    ts_text      = format_index_for_prompt(ts_subset)
    ts_text      = truncate_text(ts_text, 400)

    input_prompt = (
        "You are a senior software engineer planning a code patch for an open GitHub issue.\n\n"
        f"ISSUE: {title}\n"
        f"LABELS: {labels}\n"
        f"ASSIGNEES: {assignee_cnt}\n"
        f"BODY: {body}\n"
        f"COMMENTS: {comment_text[:600] if comment_text else 'none'}\n"
        f"REPO_SYMBOLS:\n{ts_text}\n\n"
        "First decide if a code change is needed. If yes, produce a structured patch plan."
    )

    # Derive reason from labels or state_reason
    # The model learns from the language of the issue, not from these labels
    # but we use labels to write a plausible reason for the NO label
    label_list = [l.get('name','') for l in issue.get('labels', [])]
    state_reason = issue.get('state_reason', '')

    if any(kw in ' '.join(label_list).lower() for kw in ['doc', 'question', 'duplicate']):
        reason = "Issue requests documentation, clarification, or is a duplicate — no code defect present."
    elif state_reason == 'not_planned':
        reason = "Issue was closed as not planned — no code change was implemented."
    else:
        reason = "Issue was resolved without a code change — likely a configuration, documentation, or support question."

    target_output = (
        "REQUIRES_CODE_CHANGE: NO\n"
        "CONFIDENCE: HIGH\n"
        f"REASON: {reason}"
    )

    full_text   = f"<|im_start|>user\n{input_prompt}<|im_end|>\n<|im_start|>assistant\n{target_output}<|im_end|>"
    token_count = count_tokens(full_text)

    if token_count > MAX_SEQ_LEN:
        return None

    return {
        'issue_number': row.ISSUE_NUMBER,
        'pr_number':    None,
        'label':        'NO',
        'input':        input_prompt,
        'output':       target_output,
        'full_text':    full_text,
        'token_count':  token_count,
        'changed_files': 0,
    }


print("Building YES pairs...")
yes_pairs = []
yes_skipped = 0
for row in yes_df.itertuples():
    pair = build_yes_pair(row)
    if pair:
        yes_pairs.append(pair)
    else:
        yes_skipped += 1
    if len(yes_pairs) % 100 == 0 and yes_pairs:
        print(f"  YES pairs built: {len(yes_pairs)} (skipped: {yes_skipped})")

print(f"\nTotal YES pairs: {len(yes_pairs)} (skipped {yes_skipped})")

print("\nBuilding NO pairs...")
no_pairs = []
no_skipped = 0
for row in no_df.itertuples():
    pair = build_no_pair(row)
    if pair:
        no_pairs.append(pair)
    else:
        no_skipped += 1
    if len(no_pairs) >= NO_TARGET:
        break

print(f"Total NO pairs: {len(no_pairs)} (skipped {no_skipped})")

In [ ]:
# ============================================================
# CELL 10 — BALANCE AND ASSEMBLE FINAL TRAINING SET
# Cap YES at YES_TARGET, take all NO (300)
# Shuffle with fixed seed for reproducibility
# Print token distribution stats
# ============================================================

# Cap YES examples
random.shuffle(yes_pairs)
yes_final = yes_pairs[:YES_TARGET]

# Take all NO examples (already capped at NO_TARGET in Cell 9)
no_final = no_pairs[:NO_TARGET]

# Combine and shuffle
all_pairs = yes_final + no_final
random.shuffle(all_pairs)

print(f"Final training set: {len(all_pairs)} examples")
print(f"  YES: {len(yes_final)} ({len(yes_final)/len(all_pairs)*100:.1f}%)")
print(f"  NO:  {len(no_final)} ({len(no_final)/len(all_pairs)*100:.1f}%)")

# Token distribution
token_counts = [p['token_count'] for p in all_pairs]
print(f"\nToken distribution:")
print(f"  min: {min(token_counts)}")
print(f"  median: {sorted(token_counts)[len(token_counts)//2]}")
print(f"  p90: {sorted(token_counts)[int(len(token_counts)*0.9)]}")
print(f"  max: {max(token_counts)}")
print(f"  over limit ({MAX_SEQ_LEN}): {sum(1 for t in token_counts if t > MAX_SEQ_LEN)} (should be 0)")

# Verify no token exceeds limit — hard assert
assert all(t <= MAX_SEQ_LEN for t in token_counts), \
    f"BUG: {sum(1 for t in token_counts if t > MAX_SEQ_LEN)} examples exceed {MAX_SEQ_LEN} tokens"
print("\nAssertion passed: all examples within token limit.")

In [ ]:
# ============================================================
# CELL 11 — QUALITY CHECKS
# Print sample YES and NO pairs for visual inspection
# Check no closure fields leaked into inputs
# ============================================================

print("=" * 60)
print("SAMPLE YES PAIR")
print("=" * 60)
sample_yes = [p for p in all_pairs if p['label'] == 'YES'][0]
print(f"Issue #{sample_yes['issue_number']} | PR #{sample_yes['pr_number']}")
print(f"Tokens: {sample_yes['token_count']} | Files changed: {sample_yes['changed_files']}")
print("\n--- INPUT (first 600 chars) ---")
print(sample_yes['input'][:600])
print("\n--- OUTPUT ---")
print(sample_yes['output'])

print()
print("=" * 60)
print("SAMPLE NO PAIR")
print("=" * 60)
sample_no = [p for p in all_pairs if p['label'] == 'NO'][0]
print(f"Issue #{sample_no['issue_number']}")
print(f"Tokens: {sample_no['token_count']}")
print("\n--- INPUT (first 600 chars) ---")
print(sample_no['input'][:600])
print("\n--- OUTPUT ---")
print(sample_no['output'])

# Check for closure field leakage in all inputs
CLOSURE_SIGNALS = ['closed_at', 'state_reason', 'closed_by', '"state": "closed"', 'resolution_days']
leaks = []
for pair in all_pairs:
    for sig in CLOSURE_SIGNALS:
        if sig.lower() in pair['input'].lower():
            leaks.append((pair['issue_number'], sig))

if leaks:
    print(f"\nWARNING: {len(leaks)} closure field leaks detected!")
    for num, sig in leaks[:5]:
        print(f"  Issue #{num}: found '{sig}'")
else:
    print("\nClosure field check passed — no leakage detected.")

In [ ]:
# ============================================================
# CELL 12 — SAVE JSONL AND CSV BACKUP
# JSONL is used for training
# CSV is a human-readable backup you can download
# ============================================================

# Write JSONL for training
with open(JSONL_PATH, 'w') as f:
    for pair in all_pairs:
        # Training JSONL only needs the full_text field (SFTTrainer reads this)
        f.write(json.dumps({'text': pair['full_text']}) + '\n')

print(f"Training JSONL saved: {JSONL_PATH} ({len(all_pairs)} examples)")

# Write CSV backup with all fields for inspection
csv_rows = []
for pair in all_pairs:
    csv_rows.append({
        'issue_number':  pair['issue_number'],
        'pr_number':     pair['pr_number'],
        'label':         pair['label'],
        'token_count':   pair['token_count'],
        'changed_files': pair['changed_files'],
        'input_preview': pair['input'][:300].replace('\n', ' '),
        'output':        pair['output'].replace('\n', ' | '),
    })

with open(CSV_BACKUP_PATH, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=csv_rows[0].keys())
    writer.writeheader()
    writer.writerows(csv_rows)

print(f"CSV backup saved: {CSV_BACKUP_PATH}")
print("Download planner_pairs.csv from the Colab file browser for backup.")

In [ ]:
# ============================================================
# CELL 13 — TRAIN / EVAL SPLIT
# 90% train, 10% eval
# Stratified so both splits have the same YES/NO ratio
# ============================================================

from sklearn.model_selection import train_test_split

labels = [p['label'] for p in all_pairs]

train_pairs, eval_pairs = train_test_split(
    all_pairs,
    test_size=0.10,
    stratify=labels,
    random_state=SEED
)

print(f"Train: {len(train_pairs)} examples")
print(f"  YES: {sum(1 for p in train_pairs if p['label']=='YES')}")
print(f"  NO:  {sum(1 for p in train_pairs if p['label']=='NO')}")
print(f"Eval: {len(eval_pairs)} examples")
print(f"  YES: {sum(1 for p in eval_pairs if p['label']=='YES')}")
print(f"  NO:  {sum(1 for p in eval_pairs if p['label']=='NO')}")

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_list([{'text': p['full_text']} for p in train_pairs])
eval_dataset  = Dataset.from_list([{'text': p['full_text']} for p in eval_pairs])

print(f"\nDatasets created.")

In [ ]:
# ============================================================
# CELL 14 — LOAD MODEL AND APPLY LORA
# Qwen2.5-Coder-7B-Instruct in bfloat16 (standard LoRA, not QLoRA)
# A100 80GB has plenty of VRAM — no quantization needed
# ============================================================

print(f"Loading base model: {BASE_MODEL}")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,   # full precision base, not quantized
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False    # required for gradient checkpointing

# Apply LoRA — Planner adapter uses r=32 as per doc
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODS,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Verify VRAM usage
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    print(f"\nVRAM: {allocated:.1f}GB allocated, {reserved:.1f}GB reserved")

In [ ]:
# ============================================================
# CELL 15 — TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    bf16=True,                        # bfloat16 training on A100
    gradient_checkpointing=True,      # save VRAM
    evaluation_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,               # keep last 3 checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=25,
    report_to="none",                 # disable wandb
    seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

print("Training arguments configured.")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Steps per epoch: ~{len(train_pairs) // (BATCH_SIZE * GRAD_ACCUM)}")
print(f"  Total steps: ~{len(train_pairs) * EPOCHS // (BATCH_SIZE * GRAD_ACCUM)}")

In [ ]:
# ============================================================
# CELL 16 — CUSTOM TRAINER WITH NO EXAMPLE LOSS WEIGHTING
# Applies 2.5x loss weight to NO examples to compensate for
# the 90-10 YES/NO imbalance without oversampling
# ============================================================

from torch.nn import CrossEntropyLoss

class WeightedSFTTrainer(SFTTrainer):
    """
    SFTTrainer subclass that applies higher loss weight to NO examples.
    Detects NO examples by checking if 'REQUIRES_CODE_CHANGE: NO' appears
    in the decoded input text.
    """

    def compute_loss(self, model, inputs, return_outputs=False):
        # Decode input_ids to check if this is a NO example
        input_ids = inputs.get("input_ids")
        decoded   = self.tokenizer.batch_decode(input_ids, skip_special_tokens=True)

        # Build per-example weight tensor (1.0 for YES, NO_LOSS_WEIGHT for NO)
        weights = []
        for text in decoded:
            if "REQUIRES_CODE_CHANGE: NO" in text:
                weights.append(NO_LOSS_WEIGHT)
            else:
                weights.append(1.0)

        outputs = model(**inputs)
        logits  = outputs.logits
        labels  = inputs.get("labels")

        # Compute per-token loss, then apply per-example weight
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        loss_fct     = CrossEntropyLoss(reduction='none', ignore_index=-100)
        per_token    = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1)
        )
        per_token    = per_token.view(shift_labels.size())

        weight_tensor = torch.tensor(
            weights, dtype=per_token.dtype, device=per_token.device
        ).unsqueeze(1)

        # Mask padding, apply weight, mean over non-padding tokens
        mask         = (shift_labels != -100).float()
        weighted     = per_token * weight_tensor * mask
        loss         = weighted.sum() / mask.sum().clamp(min=1)

        return (loss, outputs) if return_outputs else loss


print("WeightedSFTTrainer defined.")

In [ ]:
# ============================================================
# CELL 17 — INITIALISE AND RUN TRAINER
# Estimated A100 time: 10-14 hours for 3000 examples, 3 epochs
# ============================================================

# Set pad token (required for batch training)
TOKENIZER.pad_token = TOKENIZER.eos_token
TOKENIZER.padding_side = "right"

trainer = WeightedSFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=TOKENIZER,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=False,   # keep examples separate — important for weighted loss
)

print("Starting training...")
print(f"  Train examples: {len(train_dataset)}")
print(f"  Eval examples:  {len(eval_dataset)}")
print(f"  Epochs: {EPOCHS}")
print(f"  Estimated time: 10-14 hours on A100")

trainer.train()

In [ ]:
# ============================================================
# CELL 18 — SAVE ADAPTER AND TOKENIZER
# Saves the LoRA adapter only (not the full 7B model)
# Upload the adapter folder to GCS/Drive for safekeeping
# ============================================================

adapter_path = f"{OUTPUT_DIR}/planner_lora_adapter"
model.save_pretrained(adapter_path)
TOKENIZER.save_pretrained(adapter_path)

print(f"LoRA adapter saved to: {adapter_path}")

# List saved files
import os
for f in os.listdir(adapter_path):
    size_mb = os.path.getsize(os.path.join(adapter_path, f)) / (1024*1024)
    print(f"  {f}: {size_mb:.1f} MB")

In [ ]:
# ============================================================
# CELL 19 — QUICK INFERENCE TEST
# Run a single issue through the trained Planner and print output
# Verify: REQUIRES_CODE_CHANGE header present, file paths look real,
# no closure fields visible, output is coherent
# ============================================================

from peft import PeftModel

# Use the first eval example as test
test_pair    = eval_pairs[0]
test_input   = test_pair['input']
test_expected = test_pair['output']

print(f"Test issue #{test_pair['issue_number']}")
print(f"Expected output:\n{test_expected}\n")
print("-" * 60)

# Tokenize
prompt_text  = f"<|im_start|>user\n{test_input}<|im_end|>\n<|im_start|>assistant\n"
input_ids    = TOKENIZER.encode(prompt_text, return_tensors="pt").to(model.device)

model.eval()
with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=300,
        temperature=0.1,
        do_sample=False,     # greedy for evaluation
        pad_token_id=TOKENIZER.eos_token_id,
    )

# Decode only the generated tokens (not the prompt)
generated   = output_ids[0][input_ids.shape[1]:]
output_text = TOKENIZER.decode(generated, skip_special_tokens=True)

print(f"Generated output:\n{output_text}")

# Checks
print("\n--- CHECKS ---")
print(f"Starts with REQUIRES_CODE_CHANGE: {'YES' if output_text.startswith('REQUIRES_CODE_CHANGE') else 'NO — FAIL'}")
print(f"Contains PLAN section: {'YES' if 'PLAN:' in output_text or 'REQUIRES_CODE_CHANGE: NO' in output_text else 'NO — check output'}")
print(f"Contains closure leakage: {'YES — FAIL' if any(s in output_text for s in ['closed_at', 'state_reason']) else 'NO — OK'}")